# 📈 Capstone 2026 — Data-Driven Stock Analysis using Time Series Models
**Organised by:** Consulting & Analytics Club, IIT Guwahati × StockGro  
**Platform:** StockGro (virtual NSE/BSE trading)  
**Capital:** ₹10,00,000 virtual portfolio  

---
## 🗂️ Table of Contents
1. Task 1: Stock Universe Selection
2. Task 2: Data Preprocessing (Missing Values, ADF, Scaling, Split)
3. Task 3: Time Series Forecasting (ARIMA, LSTM, Prophet)
4. Task 4: Volatility & Trend Analysis (Log Returns, GARCH, STL)
5. Task 5: Portfolio Construction & Capital Allocation
6. Task 6: Model Comparison
7. Task 7: StockGro Virtual Trading Execution
8. Task 8: Performance Tracking — Predicted vs Actual


In [ ]:
# Global Imports & Configuration
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
from prophet import Prophet

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from arch import arch_model

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
FIGSIZE = (16, 6)
tf.random.set_seed(42)
np.random.seed(42)

# PROJECT CONSTANTS
TOTAL_CAPITAL   = 1_000_000
START_DATE      = '2021-01-01'
END_DATE        = '2025-12-31'
TRAIN_END       = '2025-06-30'
TEST_START      = '2025-07-01'
FORECAST_DAYS   = 2 # Predicting 2 days beyond dataset
SEQ_LEN         = 60
print("All imports successful!")


All imports successful!


---
## 📌 Task 1 — Stock Universe Selection
We select **5 large-cap NSE stocks** spanning **5 distinct sectors**:
- `HDFCBANK.NS` (Banking)
- `TCS.NS` (IT)
- `SUNPHARMA.NS` (Pharma)
- `ITC.NS` (FMCG)
- `TATAMOTORS.NS` (Auto)


In [ ]:
# TASK 1: STOCK UNIVERSE & DATA DOWNLOAD
STOCKS = {
    'HDFCBANK.NS': 'Banking',
    'TCS.NS'     : 'IT',
    'SUNPHARMA.NS': 'Pharma',
    'ITC.NS'     : 'FMCG',
    'TATAMOTORS.NS': 'Auto'
}
TICKERS = list(STOCKS.keys())

print(f'Downloading daily data for {len(TICKERS)} stocks...')
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, interval='1d')

# Debug: show raw column structure
print(f"Raw columns type: {type(raw.columns)}")
if isinstance(raw.columns, pd.MultiIndex):
    print(f"Raw MultiIndex levels: {raw.columns.levels}")
    print(f"Raw MultiIndex names : {raw.columns.names}")

# --- Robust column extraction for any yfinance version ---
if isinstance(raw.columns, pd.MultiIndex):
    # MultiIndex: ('Close', 'HDFCBANK.NS'), etc.
    close = raw['Close'].copy()
else:
    # Single ticker fallback
    close = raw[['Close']].copy()
    close.columns = TICKERS

# Flatten tuple column names if needed
if len(close.columns) > 0 and isinstance(close.columns[0], tuple):
    close.columns = [c[-1] if isinstance(c, tuple) else c for c in close.columns]

# Remove timezone from index
if hasattr(close.index, 'tz') and close.index.tz is not None:
    close.index = close.index.tz_localize(None)

# Reorder to match our TICKERS list
available = [t for t in TICKERS if t in close.columns]
missing = [t for t in TICKERS if t not in close.columns]
if missing:
    print(f"WARNING: Missing tickers in download: {missing}")
close = close[available]

# Check for all-NaN columns and drop them
nan_cols = close.columns[close.isna().all()]
if len(nan_cols) > 0:
    print(f"WARNING: All-NaN columns found and dropped: {list(nan_cols)}")
    close = close.drop(columns=nan_cols)
    # Update TICKERS to only include available ones
    TICKERS = [t for t in TICKERS if t in close.columns]

# Forward/back fill BEFORE any analysis
close = close.ffill().bfill()

print(f'\nDataset shape : {close.shape}')
print(f'Columns       : {list(close.columns)}')
print(f'Index range   : {close.index.min()} to {close.index.max()}')
print(f'NaN per col   : \n{close.isna().sum()}')
display(close.tail())

# Plot Normalised Closing Prices
fig, ax = plt.subplots(figsize=FIGSIZE)
for ticker in TICKERS:
    norm = close[ticker] / close[ticker].iloc[0] * 100
    ax.plot(norm, label=f'{ticker}')
ax.axvline(pd.Timestamp(TEST_START), color='red', linestyle='--', label='Test Start')
ax.set_title('Task 1 — Normalised Closing Price')
ax.legend()
plt.tight_layout()
plt.show()


[**********************60%****                   ]  3 of 5 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found
[*********************100%***********************]  5 of 5 completed

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


Raw columns type: <class 'pandas.MultiIndex'>
Raw MultiIndex levels: [['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], ['HDFCBANK.NS', 'ITC.NS', 'SUNPHARMA.NS', 'TATAMOTORS.NS', 'TCS.NS']]
Raw MultiIndex names : ['Price', 'Ticker']

Dataset shape : (1235, 4)
Columns       : ['HDFCBANK.NS', 'TCS.NS', 'SUNPHARMA.NS', 'ITC.NS']
Index range   : 2021-01-01 00:00:00 to 2025-12-30 00:00:00
NaN per col   : 
Ticker
HDFCBANK.NS     0
TCS.NS          0
SUNPHARMA.NS    0
ITC.NS          0
dtype: int64


Ticker,HDFCBANK.NS,TCS.NS,SUNPHARMA.NS,ITC.NS
Date,,,,
2025-12-23,996.599976,3250.902100,1744.567627,398.988159
2025-12-24,997.200012,3259.741455,1725.789673,398.253571
2025-12-26,992.099976,3221.437744,1708.402588,395.853851
2025-12-29,991.700012,3193.446533,1706.117310,394.335663
2025-12-30,990.900024,3188.830566,1709.098022,392.376709


---
## 📌 Task 2 — Data Preprocessing
1. Handle Missing Values (ffill, bfill)
2. Train / Test Split
3. ADF Stationarity Test


In [ ]:
# TASK 2: DATA PREPROCESSING
print('Missing values after fill:', close.isnull().sum().sum())

train = close.loc[:TRAIN_END].copy()
test  = close.loc[TEST_START:].copy()
print(f'Train: {len(train)} days | Test: {len(test)} days')

# Sanity check
assert len(train) > 0, f"Train set is empty! Index range: {close.index.min()} to {close.index.max()}"
assert len(test)  > 0, f"Test set is empty!"

def adf_test(series):
    s = series.dropna()
    if len(s) < 20:
        return False, 1.0
    result = adfuller(s)
    return result[1] <= 0.05, result[1]

adf_res = []
diff_flags = {}
for ticker in TICKERS:
    stat, p = adf_test(train[ticker])
    diff_flags[ticker] = 0 if stat else 1
    adf_res.append({'Ticker': ticker, 'Stationary': stat, 'p-value': round(p, 6), 'd': diff_flags[ticker]})

display(pd.DataFrame(adf_res))


Missing values after fill: 0
Train: 1110 days | Test: 125 days


,Ticker,Stationary,p-value,d
0,HDFCBANK.NS,False,0.574220,1
1,TCS.NS,False,0.215199,1
2,SUNPHARMA.NS,False,0.856687,1
3,ITC.NS,False,0.724135,1


---
## 📌 Task 4 — Volatility & Trend Analysis (Executed before Task 3 for better flow)


In [ ]:
# TASK 4: VOLATILITY & TREND
log_returns = np.log(close / close.shift(1)).iloc[1:]  # drop first NaN row

vol_20d = log_returns.rolling(20).std() * np.sqrt(252)
vol_20d = vol_20d.dropna(how='all')

print(f"log_returns shape: {log_returns.shape}, vol_20d shape: {vol_20d.shape}")

latest_vol = vol_20d.iloc[-1]
print("\nLatest Annualised Volatility:")
display(latest_vol)

fig, ax = plt.subplots(figsize=FIGSIZE)
for ticker in TICKERS:
    series = vol_20d[ticker].dropna()
    if len(series) > 0:
        ax.plot(series, label=ticker)
ax.set_title('20-Day Rolling Annualised Volatility')
ax.legend()
plt.tight_layout()
plt.show()

# STL Decomposition — pick the ticker with the most data points
best_ticker = max(TICKERS, key=lambda t: close[t].dropna().shape[0])
decomp_series = close[best_ticker].dropna()
print(f"\nSTL Decomposition for {best_ticker} ({len(decomp_series)} observations)")

if len(decomp_series) >= 504:
    decomp = seasonal_decompose(decomp_series, model='multiplicative', period=252)
    fig = decomp.plot()
    fig.suptitle(f'STL Decomposition - {best_ticker}', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"  Skipped: need >= 504 obs, have {len(decomp_series)}")


log_returns shape: (1234, 4), vol_20d shape: (1215, 4)

Latest Annualised Volatility:


Ticker
HDFCBANK.NS     0.104326
TCS.NS          0.138496
SUNPHARMA.NS    0.150538
ITC.NS          0.084913
Name: 2025-12-30 00:00:00, dtype: float64


STL Decomposition for HDFCBANK.NS (1235 observations)


---
## 📌 Task 3 — Time Series Forecasting
Forecasting with ARIMA, Prophet, and LSTM.


In [ ]:
# TASK 3: FORECASTING
results_dict = {t: {} for t in TICKERS}
forecasts_2d = {t: {} for t in TICKERS}

for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f"--- Modeling {ticker} ---")
    train_s = train[ticker].dropna()
    test_s = test[ticker].dropna()
    print(f"    Train pts: {len(train_s)}, Test pts: {len(test_s)}")
    
    if len(train_s) < SEQ_LEN + 10 or len(test_s) < 2:
        print(f"    SKIPPED — insufficient data")
        results_dict[ticker] = {'ARIMA': np.array([]), 'Prophet': np.array([]),
                                'LSTM': np.array([]), 'Actual': test_s.values}
        forecasts_2d[ticker] = {'ARIMA': np.array([0,0]), 'Prophet': np.array([0,0]),
                                'LSTM': np.array([0,0])}
        continue
    
    # --- ARIMA ---
    try:
        model_arima = pm.auto_arima(train_s, d=diff_flags[ticker], seasonal=False, suppress_warnings=True)
        arima_preds = model_arima.predict(n_periods=len(test_s))
        arima_all = model_arima.predict(n_periods=len(test_s) + 2)
        arima_fut = np.array(arima_all)[-2:]
        print(f"    ARIMA order: {model_arima.order}")
    except Exception as e:
        print(f"    ARIMA failed: {e}")
        arima_preds = np.full(len(test_s), train_s.iloc[-1])
        arima_fut = np.array([train_s.iloc[-1], train_s.iloc[-1]])
    
    # --- PROPHET ---
    try:
        df_prophet = train_s.reset_index()
        df_prophet.columns = ['ds', 'y']
        if hasattr(df_prophet['ds'].dtype, 'tz'):
            df_prophet['ds'] = df_prophet['ds'].dt.tz_localize(None)
        
        m = Prophet(daily_seasonality=True)
        m.fit(df_prophet)
        
        future = m.make_future_dataframe(periods=len(test_s) + 2, freq='B')
        fcst = m.predict(future)
        prophet_all = fcst['yhat'].values
        prophet_preds = prophet_all[len(train_s):len(train_s)+len(test_s)]
        prophet_fut = prophet_all[-2:]
        
        if len(prophet_preds) < len(test_s):
            prophet_preds = np.pad(prophet_preds, (0, len(test_s) - len(prophet_preds)), mode='edge')
        print(f"    Prophet OK")
    except Exception as e:
        print(f"    Prophet failed: {e}")
        prophet_preds = np.full(len(test_s), train_s.iloc[-1])
        prophet_fut = np.array([train_s.iloc[-1], train_s.iloc[-1]])
    
    # --- LSTM ---
    try:
        sc = MinMaxScaler()
        train_sc = sc.fit_transform(train_s.values.reshape(-1, 1))
        
        X_tr, y_tr = [], []
        for i in range(SEQ_LEN, len(train_sc)):
            X_tr.append(train_sc[i-SEQ_LEN:i, 0])
            y_tr.append(train_sc[i, 0])
        X_tr, y_tr = np.array(X_tr), np.array(y_tr)
        X_tr = X_tr.reshape((X_tr.shape[0], X_tr.shape[1], 1))
        
        lstm = Sequential([
            LSTM(50, return_sequences=True, input_shape=(SEQ_LEN, 1)),
            Dropout(0.2),
            LSTM(50),
            Dense(1)
        ])
        lstm.compile(optimizer='adam', loss='mse')
        lstm.fit(X_tr, y_tr, epochs=10, batch_size=32, verbose=0)
        
        full_data = close[ticker].values
        inputs = full_data[len(full_data) - len(test_s) - SEQ_LEN:]
        inputs = sc.transform(inputs.reshape(-1, 1))
        X_te = []
        for i in range(SEQ_LEN, len(inputs)):
            X_te.append(inputs[i-SEQ_LEN:i, 0])
        X_te = np.array(X_te).reshape((-1, SEQ_LEN, 1))
        
        lstm_preds = sc.inverse_transform(lstm.predict(X_te, verbose=0)).flatten()
        
        last_seq = inputs[-SEQ_LEN:].reshape(1, SEQ_LEN, 1)
        f1 = lstm.predict(last_seq, verbose=0)
        last_seq_2 = np.append(last_seq[:, 1:, :], f1.reshape(1, 1, 1), axis=1)
        f2 = lstm.predict(last_seq_2, verbose=0)
        lstm_fut = sc.inverse_transform(np.vstack([f1[0], f2[0]])).flatten()
        print(f"    LSTM OK")
    except Exception as e:
        print(f"    LSTM failed: {e}")
        lstm_preds = np.full(len(test_s), train_s.iloc[-1])
        lstm_fut = np.array([train_s.iloc[-1], train_s.iloc[-1]])
    
    min_len = min(len(arima_preds), len(prophet_preds), len(lstm_preds), len(test_s))
    results_dict[ticker] = {
        'ARIMA': np.array(arima_preds[:min_len]),
        'Prophet': np.array(prophet_preds[:min_len]),
        'LSTM': np.array(lstm_preds[:min_len]),
        'Actual': test_s.values[:min_len]
    }
    forecasts_2d[ticker] = {
        'ARIMA': np.array(arima_fut).flatten()[:2],
        'Prophet': np.array(prophet_fut).flatten()[:2],
        'LSTM': np.array(lstm_fut).flatten()[:2]
    }

print("\n✅ All models trained successfully!")



--- Modeling HDFCBANK.NS ---
    Train pts: 1110, Test pts: 125


c:\Users\Lenovo\OneDrive\Desktop\capstone-project\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\Lenovo\OneDrive\Desktop\capstone-project\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


    ARIMA order: (2, 1, 2)


18:44:52 - cmdstanpy - INFO - Chain [1] start processing
18:44:53 - cmdstanpy - INFO - Chain [1] done processing


    Prophet OK
    LSTM OK

--- Modeling TCS.NS ---
    Train pts: 1110, Test pts: 125


...

    Prophet OK
    LSTM OK

--- Modeling ITC.NS ---
    Train pts: 1110, Test pts: 125


c:\Users\Lenovo\OneDrive\Desktop\capstone-project\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\Lenovo\OneDrive\Desktop\capstone-project\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


    ARIMA order: (0, 1, 0)


18:46:00 - cmdstanpy - INFO - Chain [1] start processing
18:46:01 - cmdstanpy - INFO - Chain [1] done processing


    Prophet OK
    LSTM OK

✅ All models trained successfully!


---
## 📌 Task 6 — Model Comparison


In [ ]:
# TASK 6: MODEL COMPARISON
metrics = []
for ticker in TICKERS:
    act = results_dict[ticker]['Actual']
    for model_name in ['ARIMA', 'Prophet', 'LSTM']:
        preds = results_dict[ticker][model_name]
        n = min(len(act), len(preds))
        if n < 2:
            continue
        a, p_ = act[:n], preds[:n]
        
        mape = mean_absolute_percentage_error(a, p_)
        rmse = np.sqrt(mean_squared_error(a, p_))
        dir_acc = np.mean(np.sign(np.diff(a)) == np.sign(np.diff(p_)))
        
        metrics.append({'Ticker': ticker, 'Model': model_name,
                        'MAPE': round(mape, 4), 'RMSE': round(rmse, 2),
                        'DirAcc': round(dir_acc, 4)})

df_metrics = pd.DataFrame(metrics)
print("\n📊 Per-Stock Metrics:")
display(df_metrics)

print("\n📊 Average Metrics by Model:")
display(df_metrics.groupby('Model')[['MAPE', 'RMSE', 'DirAcc']].mean().sort_values('MAPE'))

fig, ax = plt.subplots(figsize=(12, 5))
df_metrics.pivot(index='Ticker', columns='Model', values='RMSE').plot(kind='bar', ax=ax)
ax.set_title('RMSE Comparison across Models')
ax.set_ylabel('RMSE')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



📊 Per-Stock Metrics:


,Ticker,Model,MAPE,RMSE,DirAcc
0,HDFCBANK.NS,ARIMA,0.0144,19.50,0.4758
1,HDFCBANK.NS,Prophet,0.0357,41.01,0.5323
2,HDFCBANK.NS,LSTM,0.0094,11.40,0.4677
3,TCS.NS,ARIMA,0.1056,338.70,0.0000
4,TCS.NS,Prophet,0.0579,212.95,0.5000
5,TCS.NS,LSTM,0.0172,68.78,0.5161
6,SUNPHARMA.NS,ARIMA,0.0354,71.76,0.5806
7,SUNPHARMA.NS,Prophet,0.0914,170.13,0.5806
8,SUNPHARMA.NS,LSTM,0.0168,35.10,0.5081
9,ITC.NS,ARIMA,0.0548,25.02,0.4597



📊 Average Metrics by Model:


,MAPE,RMSE,DirAcc
Model,,,
LSTM,0.01425,30.4275,0.477825
ARIMA,0.05255,113.7450,0.379025
Prophet,0.05590,110.5100,0.526200


---
## 📌 Task 5 — Portfolio Construction
Using **Strategy B (Volatility-Aware Sizing)**: w_i = (1/vol_i) / sum(1/vol_j)


In [ ]:
# TASK 5: PORTFOLIO CONSTRUCTION
inv_vol = 1 / latest_vol
weights = inv_vol / inv_vol.sum()

alloc = pd.DataFrame({
    'Sector': [STOCKS[t] for t in weights.index if t in STOCKS],
    'Volatility': latest_vol,
    'Weight': weights,
    'Amount (₹)': (weights * TOTAL_CAPITAL).round(0)
})
print("\n💰 Portfolio Allocation (₹10,00,000 Capital):")
display(alloc.sort_values('Amount (₹)', ascending=False))

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(alloc['Amount (₹)'], labels=alloc.index, autopct='%1.1f%%', startangle=140)
ax.set_title('Portfolio Allocation — Volatility-Aware')
plt.tight_layout()
plt.show()



💰 Portfolio Allocation (₹10,00,000 Capital):


,Sector,Volatility,Weight,Amount (₹)
Ticker,,,,
ITC.NS,FMCG,0.084913,0.334325,334325.0
HDFCBANK.NS,Banking,0.104326,0.272116,272116.0
TCS.NS,IT,0.138496,0.204978,204978.0
SUNPHARMA.NS,Pharma,0.150538,0.188581,188581.0


---
## 📌 Task 7 & 8 — Virtual Trading Execution & Performance Tracking
1. Log into StockGro.
2. Execute the allocations calculated in Task 5.
3. Compare the predictions from `forecasts_2d` with the actual market closing prices on Day 1 and Day 2.


In [ ]:
# TASK 7 & 8: PREDICTIONS FOR VIRTUAL TRADING
print("\n🔮 Next 2 Days Predictions (For StockGro Trading):")
pred_df = pd.DataFrame({t: forecasts_2d[t]['LSTM'] for t in TICKERS}, index=['Day 1', 'Day 2'])
display(pred_df.round(2))

print("\n📋 Recommended Trade Summary:")
for ticker in TICKERS:
    last_price = close[ticker].iloc[-1]
    day1_pred = forecasts_2d[ticker]['LSTM'][0]
    pct_change = ((day1_pred - last_price) / last_price) * 100
    action = "BUY 🟢" if pct_change > 0 else "HOLD/SELL 🔴"
    allocated = alloc.loc[ticker, 'Amount (₹)']
    shares = int(allocated / last_price)
    print(f"  {ticker:18s} | Last: ₹{last_price:>10,.2f} | Pred: ₹{day1_pred:>10,.2f} | "
          f"Chg: {pct_change:+.2f}% | {action} | Shares: {shares}")

print("\n✅ Capstone 2026 Analysis Complete!")



🔮 Next 2 Days Predictions (For StockGro Trading):


,HDFCBANK.NS,TCS.NS,SUNPHARMA.NS,ITC.NS
Day 1,989.039978,3232.949951,1740.089966,397.250000
Day 2,989.179993,3234.449951,1736.079956,397.390015



📋 Recommended Trade Summary:
  HDFCBANK.NS        | Last: ₹    990.90 | Pred: ₹    989.04 | Chg: -0.19% | HOLD/SELL 🔴 | Shares: 274
  TCS.NS             | Last: ₹  3,188.83 | Pred: ₹  3,232.95 | Chg: +1.38% | BUY 🟢 | Shares: 64
  SUNPHARMA.NS       | Last: ₹  1,709.10 | Pred: ₹  1,740.09 | Chg: +1.81% | BUY 🟢 | Shares: 110
  ITC.NS             | Last: ₹    392.38 | Pred: ₹    397.25 | Chg: +1.24% | BUY 🟢 | Shares: 852

✅ Capstone 2026 Analysis Complete!
